In [ ]:
"""
Parte 2 — Melhorando o método ingênuo
Três classes (fundo / interior / fronteira) + mapa de distância ao fundo,
decodificados por watershed marcado. Reaproveita partes da Parte 1
(mask_iou, greedy_match, IOU_THRESHOLDS) — redefinidas aqui para o arquivo
ficar autocontido.

model = UNetTernary()  (já criado por vocês)

ATENÇÃO / PREMISSA A AJUSTAR:
  Como classificação ternária e regressão de distância são duas tarefas
  (multi-task), assumo que UNetTernary.forward(images) devolve uma tupla:
      logits_class : (B, 3, H, W)  -> fundo/interior/fronteira
      dist_pred    : (B, 1, H, W)  -> mapa de distância contínuo, sem ativação
  Se a arquitetura de vocês só tem a cabeça de classes (sem regressão de
  distância), ajustem `forward_pass()` abaixo e removam os termos de
  `dist_pred` na loss e na decodificação (watershed cai de volta a usar
  -prob_interior como elevação, o que também funciona, só que pior).
"""


# ---------------------------------------------------------------------------
# 7. Decodificação por watershed marcado
# ---------------------------------------------------------------------------

def decode_watershed(class_probs, dist_pred=None, interior_threshold=0.5,
                      fg_threshold=0.5, min_marker_size=5):
    """
    class_probs: (3,H,W) softmax das classes [fundo, interior, fronteira].
    dist_pred:   (H,W) opcional, mapa de distância previsto.

    1. MARCADORES = componentes conexas da classe "interior" binarizada
       (prob_interior > interior_threshold). Cada componente vira a
       semente de uma futura instância — é isto que faz o watershed ser
       "marcado" e não sofrer de over-segmentation like watershed clássico.
    2. MÁSCARA de foreground = tudo que não é fundo (interior + fronteira),
       impede o watershed de "vazar" e invadir o fundo.
    3. SUPERFÍCIE DE ELEVAÇÃO = -dist_pred (se disponível) ou -prob_interior
       como fallback. O watershed "inunda" a partir dos marcadores subindo
       essa superfície; como o centro dos núcleos tem distância maior
       (elevação mais baixa = "vale") e as bordas/fronteiras têm distância
       menor (elevação mais alta = "montanha"), a água de duas instâncias
       vizinhas se encontra exatamente na fronteira entre elas, separando-as.
    """
    prob_fundo, prob_interior, prob_fronteira = class_probs

    markers_bin = prob_interior > interior_threshold
    markers, num_markers = ndimage.label(markers_bin, structure=np.ones((3, 3)))
    for i in range(1, num_markers + 1):
        if (markers == i).sum() < min_marker_size:
            markers[markers == i] = 0

    foreground_mask = (prob_interior + prob_fronteira) > fg_threshold

    elevation = -dist_pred if dist_pred is not None else -prob_interior

    labels_ws = watershed(elevation, markers=markers, mask=foreground_mask)

    instances = []
    for i in range(1, labels_ws.max() + 1):
        inst = (labels_ws == i).astype(np.uint8)
        if inst.sum() > 0:
            instances.append(inst)
    return instances


# ---------------------------------------------------------------------------
# 8. Avaliação de instâncias (reaproveita a MESMA regra de matching da
#    Parte 1 — guloso por IoU decrescente — para os números ficarem
#    comparáveis entre o método ingênuo e este)
# ---------------------------------------------------------------------------

def mask_iou(m1, m2):
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    return inter / union if union > 0 else 0.0

def greedy_match(pred_masks, gt_masks, iou_threshold):
    n_pred, n_gt = len(pred_masks), len(gt_masks)
    if n_pred == 0 and n_gt == 0:
        return 0, 0, 0
    if n_pred == 0:
        return 0, 0, n_gt
    if n_gt == 0:
        return 0, n_pred, 0
    iou_matrix = np.zeros((n_pred, n_gt))
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            iou_matrix[i, j] = mask_iou(pm, gm)
    pairs = sorted(((iou_matrix[i, j], i, j) for i in range(n_pred) for j in range(n_gt)),
                    key=lambda x: -x[0])
    matched_pred, matched_gt, tp = set(), set(), 0
    for iou, i, j in pairs:
        if iou < iou_threshold:
            break
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(i); matched_gt.add(j); tp += 1
    return tp, n_pred - tp, n_gt - tp

IOU_THRESHOLDS = np.arange(0.50, 1.00, 0.05)

@torch.no_grad()
def evaluate_instances_watershed(model, loader, interior_threshold=0.5,
                                  fg_threshold=0.5, min_marker_size=5):
    model.eval()
    agg = {t: {'tp': 0, 'fp': 0, 'fn': 0} for t in IOU_THRESHOLDS}
    per_image_records = []

    for images, labels, dists, instance_masks_batch, ids in tqdm(loader, desc='Avaliando (watershed)'):
        images = images.to(DEVICE)
        logits_class, dist_pred = forward_pass(model, images)
        probs = F.softmax(logits_class, dim=1).cpu().numpy()          # (B,3,H,W)
        dist_np = dist_pred.cpu().numpy() if dist_pred is not None else None

        for b in range(images.size(0)):
            class_probs = probs[b]
            d_map = dist_np[b, 0] if dist_np is not None else None
            gt_masks = instance_masks_batch[b]

            pred_masks = decode_watershed(class_probs, d_map,
                                           interior_threshold, fg_threshold,
                                           min_marker_size)

            n_gt, n_pred = len(gt_masks), len(pred_masks)
            count_error = abs(n_pred - n_gt)

            ap_per_threshold = []
            for t in IOU_THRESHOLDS:
                tp, fp, fn = greedy_match(pred_masks, gt_masks, t)
                agg[t]['tp'] += tp; agg[t]['fp'] += fp; agg[t]['fn'] += fn
                denom = tp + fp + fn
                ap_per_threshold.append(tp / denom if denom > 0 else 1.0)

            per_image_records.append({
                'id': ids[b], 'n_gt': n_gt, 'n_pred': n_pred,
                'count_error': count_error,
                'mAP_image': float(np.mean(ap_per_threshold)),
            })

    ap_by_threshold = {}
    for t in IOU_THRESHOLDS:
        tp, fp, fn = agg[t]['tp'], agg[t]['fp'], agg[t]['fn']
        denom = tp + fp + fn
        ap_by_threshold[round(t, 2)] = tp / denom if denom > 0 else 1.0
    mAP = float(np.mean(list(ap_by_threshold.values())))

    df = pd.DataFrame(per_image_records)
    return {
        'ap_by_threshold': ap_by_threshold,
        'mAP': mAP,
        'mean_count_error': df['count_error'].mean(),
        'per_image_df': df,
    }


# ---------------------------------------------------------------------------
# Execução
# ---------------------------------------------------------------------------



results = evaluate_instances_watershed(model, val_loader)

print('\nAP por limiar de IoU (watershed):')
for t, ap in results['ap_by_threshold'].items():
	print(f'  IoU={t:.2f}: AP={ap:.4f}')
print(f"\nmAP (0.50:0.95): {results['mAP']:.4f}")
print(f"Erro absoluto médio de contagem por imagem: {results['mean_count_error']:.3f}")

results['per_image_df'].to_csv('instance_eval_watershed.csv', index=False)

# comparação direta com o baseline ingênuo (Parte 1), se o CSV existir
try:
	baseline_df = pd.read_csv('instance_eval_per_image.csv')
	print(f"\nComparação:")
	print(f"  Baseline (limiar+CC): erro médio de contagem = "
			f"{baseline_df['count_error'].mean():.3f}")
	print(f"  Watershed marcado:    erro médio de contagem = "
			f"{results['mean_count_error']:.3f}")
except FileNotFoundError:
	pass

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Calculando pesos de classe (balanceamento por frequência mediana)...
Pesos [fundo, interior, fronteira] = [0.09790992736816406, 1.0, 1.6255546808242798]
<class 'tuple'>
[torch.Size([16, 3, 128, 128]), torch.Size([16, 1, 128, 128])]


Epoch 1/10: 100%|██████████| 36/36 [08:54<00:00, 14.85s/it]


  train_loss=0.1875  val_loss=0.1925 (cls=0.1284, dist=0.0641)


Epoch 2/10: 100%|██████████| 36/36 [09:04<00:00, 15.12s/it]


  train_loss=0.1309  val_loss=0.1218 (cls=0.0762, dist=0.0457)


...

  train_loss=0.0618  val_loss=0.0673 (cls=0.0395, dist=0.0278)


Epoch 10/10: 100%|██████████| 36/36 [08:48<00:00, 14.69s/it]


  train_loss=0.0571  val_loss=0.0652 (cls=0.0398, dist=0.0255)


Avaliando (watershed): 100%|██████████| 7/7 [04:56<00:00, 42.32s/it]


AP por limiar de IoU (watershed):
  IoU=0.50: AP=0.2923
  IoU=0.55: AP=0.2284
  IoU=0.60: AP=0.1729
  IoU=0.65: AP=0.1323
  IoU=0.70: AP=0.0968
  IoU=0.75: AP=0.0688
  IoU=0.80: AP=0.0369
  IoU=0.85: AP=0.0152
  IoU=0.90: AP=0.0042
  IoU=0.95: AP=0.0002

mAP (0.50:0.95): 0.1048
Erro absoluto médio de contagem por imagem: 22.110

Comparação:
  Baseline (limiar+CC): erro médio de contagem = 20.130
  Watershed marcado:    erro médio de contagem = 22.110
